# CrowdHumanでYOLO人物検出モデルをファインチューニング

CrowdHumanの **vbox（人物の見えている範囲）とhbox（頭部）** を使ってYOLOをファインチューニングします。1つのモデルで、`0: person`（vbox）と `1: head`（hbox）の2クラスを学習します。

GPU確認・Google Drive接続・2クラスのデータセット準備・事前学習モデル取得・学習・Driveへの自動保存・再開まで実装しています。上から順に実行してください。セル6を実行すると学習が開始します。

学習処理の実装方針:

- 小規模学習は挟まず、設定したエポック数で学習します。
- 各エポック終了時に再開用状態をデータ型の変換なしでDriveへ保存し、コピー確認後に再開先を更新します。
- 中断後は最後に正常保存されたエポックの次から再開します。
- 学習損失にNaN/Infを検出した場合は停止します。
- CSV・図などの学習ログをローカルに出力し、セル7でダウンロードできます。

確認済みの保存場所:

- データ: `/content/drive/MyDrive/crowdhuman`
- 注釈: `raw/annotations/annotation_train.odgt`、`raw/annotations/annotation_val.odgt`（データフォルダからの相対パス）
- 保存済みデータの管理情報: `prepared/latest.json`（データフォルダからの相対パス）
- 管理情報の参照先: `prepared_9c497bb835b6435ea77e0992ab70b7d4.tar`（`prepared` 内）。設定は `vbox`。hboxを含むか、tarの中身は未確認です。

今後使用する保存先:

- 学習ログ（ローカル）: `/content/crowdhuman_training`
- 学習結果（Drive）: `/content/drive/MyDrive/crowdhuman_vbox_ft/runs`
- 事前学習重み: `/content/drive/MyDrive/crowdhuman_vbox_ft/weights`
- ローカルデータ展開先: `/content/crowdhuman_vbox_yolo`


## 1. GPUを確認

Colabの「ランタイム」→「ランタイムのタイプを変更」でGPUを選択してから実行してください。
`nvidia-smi` の結果と、PyTorchから見えるGPU・メモリ・バージョンを表示します。
CUDAが利用できない場合は、エラーで停止します。


In [ ]:
import subprocess
import torch

try:
    result = subprocess.run(
        ["nvidia-smi"], capture_output=True, text=True, check=False
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
except FileNotFoundError:
    print("nvidia-smiが見つかりません。ColabのGPU設定を確認してください。")

print(f"PyTorch: {torch.__version__}")
print(f"PyTorchのCUDAバージョン: {torch.version.cuda}")
print(f"CUDA利用可能: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for index in range(torch.cuda.device_count()):
        properties = torch.cuda.get_device_properties(index)
        print(f"GPU {index}: {properties.name}")
        print(f"GPU {index}の総メモリ: {properties.total_memory / 1024**3:.2f} GiB")
else:
    raise RuntimeError("GPUが利用できません。Colabの「ランタイム」→「ランタイムのタイプを変更」でGPUを選択してから再実行してください。")


## 2. Google Driveを接続

画面の案内に従って、CrowdHumanを保存しているGoogleアカウントのDriveを接続してください。


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 3. データセット準備の設定と取得

Driveの `raw/archives` の4個のZIPと、`raw/annotations` のtrain・val注釈を使用します。
注釈2ファイルと画像ZIP4個がDriveに無い場合は、このセルがHugging Faceのミラーから取得します。サイズとSHA-256はミラー間で一致した値で検証します。注釈は公式のGoogle Driveも予備として使用します。
画像ZIPは合計約10.9GiBあります。中断してもセル3を再実行すれば続きから取得します。Driveの空き容量が不足している場合は、取得を始める前に停止します。
`prepared` のtarは管理情報がvbox設定のため、今回の2クラス変換には使用しません。
Driveの不足・破損しているファイルだけ取得し直し、それ以外の既存データは変更しません。ローカルへの展開とラベル変換には時間・空き容量が必要です。

出力先は今回の準備処理専用にしてください。設定が一致する途中出力は再利用します。
Colabのローカルデータが失われた場合は、セル3〜4を再実行して準備し直します。


In [ ]:
import hashlib
import json
import shutil
import subprocess
import sys
import urllib.request
import zipfile
from pathlib import Path

from tqdm.auto import tqdm

DATA_DIR = Path("/content/drive/MyDrive/crowdhuman")
OUTPUT_DIR = Path("/content/crowdhuman_vbox_yolo")
ANNOTATIONS_DIR = DATA_DIR / "raw" / "annotations"
ARCHIVE_DIR = DATA_DIR / "raw" / "archives"

if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("先にセル2でGoogle Driveを接続してください。")

# Hugging Faceのミラーは公式配布と同じLFSハッシュです。サイズとSHA-256で検証します。
MIRRORS = ("sshao0516", "Funtion")
ANNOTATION_SOURCES = {
    "annotation_train.odgt": {
        "size": 80017502,
        "sha256": "6bf241a79f19e30cf52681eab3392368bd5a534164be9272e7a808cb284d9f77",
        "gdrive_id": "1UUTea5mYqvlUObsC1Z8CFldHJAtLtMX3",
    },
    "annotation_val.odgt": {
        "size": 23323139,
        "sha256": "be422c79a190ff7e30fe5cbd74cbf45a2dadda6c5af58c6ec11a038ba2993c04",
        "gdrive_id": "10WIRwu8ju8GRLuCkZ_vT6hnNxs5ptwoL",
    },
}
ARCHIVE_SOURCES = {
    "CrowdHuman_train01.zip": {
        "size": 2970597373,
        "sha256": "7ba340163cff0f2446027af95dc96bcb9c66be18506eabb57822c400a7efd3b8",
    },
    "CrowdHuman_train02.zip": {
        "size": 3092749718,
        "sha256": "d9ecfb43eaf8381ddd4d1ff4e9a0877b694dbfba7a7a33ed3966ee2c2a628663",
    },
    "CrowdHuman_train03.zip": {
        "size": 2306357030,
        "sha256": "9cac171914c4f5c7371e3d42d20b50858caa1602811ae573aa35bbfe672dc29f",
    },
    "CrowdHuman_val.zip": {
        "size": 2488658160,
        "sha256": "c0ab99bb80ac162cd3efdf94a1a6100c4f059a61d69596412c6b44ebc20d1363",
    },
}
MANUAL_HINT = (
    f"注釈は {ANNOTATIONS_DIR}、画像ZIPは {ARCHIVE_DIR} に手動で置くか、"
    "セル3を再実行して続きから取得してください: https://www.crowdhuman.org/download.html"
)


def mirror_urls(name):
    return [f"https://huggingface.co/datasets/{repo}/CrowdHuman/resolve/main/{name}"
            for repo in MIRRORS]


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def download_resumable(url, target, spec):
    start = target.stat().st_size if target.is_file() else 0
    if start > spec["size"]:
        target.unlink()
        start = 0
    if start == spec["size"]:
        return sha256_file(target)
    digest = hashlib.sha256()
    if start:
        # 再開時は既存の.partを読み直し、ハッシュを引き継ぎます。
        with target.open("rb") as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(block)
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    if start:
        request.add_header("Range", f"bytes={start}-")
        print(f"  {target.name}: {start:,} bytes から再開")
    else:
        print(f"  {target.name}: 先頭から取得")
    response = urllib.request.urlopen(request, timeout=120)
    with response:
        if start and response.status != 206:
            # Range非対応のサーバーだった場合は最初から取り直します。
            target.unlink()
            return download_resumable(url, target, spec)
        mode = "ab" if start else "wb"
        with target.open(mode) as stream, tqdm(
                total=spec["size"], initial=start, unit="B", unit_scale=True,
                desc=target.name, leave=False) as progress:
            while True:
                block = response.read(1024 * 1024)
                if not block:
                    break
                stream.write(block)
                digest.update(block)
                progress.update(len(block))
    size = target.stat().st_size
    if size != spec["size"]:
        raise IOError(f"ダウンロードが途中で終了しました: {size:,} / {spec['size']:,} bytes")
    return digest.hexdigest()


def verify_download(path, spec, digest=None):
    size = path.stat().st_size
    if size != spec["size"]:
        raise IOError(f"サイズ不一致: {size:,} != {spec['size']:,}")
    if digest is None:
        digest = sha256_file(path)
    if digest != spec["sha256"]:
        raise ValueError(f"SHA-256不一致: {digest}")


def annotation_is_usable(path, spec):
    # 再実行を速くするため、既存ファイルはサイズと先頭行だけ確認します。
    if not path.is_file() or path.stat().st_size != spec["size"]:
        return False
    try:
        with path.open(encoding="utf-8") as stream:
            record = json.loads(stream.readline())
    except (UnicodeDecodeError, json.JSONDecodeError):
        return False
    return (isinstance(record, dict)
            and isinstance(record.get("ID"), str)
            and isinstance(record.get("gtboxes"), list))


def archive_is_usable(path, spec):
    # 既存ZIPはサイズとZIP構造だけ確認し、ハッシュは取得時に検証します。
    if not path.is_file() or path.stat().st_size != spec["size"]:
        return False
    try:
        return zipfile.is_zipfile(path)
    except OSError:
        return False


def fetch_from_gdrive(spec, target):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown
    if gdown.download(id=spec["gdrive_id"], output=str(target), quiet=True) is None:
        raise RuntimeError("ダウンロード結果が空です")


def fetch_file(name, spec, directory, is_usable):
    target = directory / name
    if is_usable(target, spec):
        print(f"取得済み: {target} ({target.stat().st_size:,} bytes)")
        return
    directory.mkdir(parents=True, exist_ok=True)
    temporary = target.with_name(target.name + ".part")
    errors = []
    for url in mirror_urls(name):
        try:
            print(f"取得中: {name}")
            digest = download_resumable(url, temporary, spec)
            verify_download(temporary, spec, digest)
        except ValueError as error:
            # 内容が一致しない.partは再開に使えません。
            errors.append(f"{url}: {error}")
            temporary.unlink(missing_ok=True)
            continue
        except Exception as error:
            # 通信エラーは.partを残し、次のミラーで続きから取得します。
            errors.append(f"{url}: {error}")
            continue
        temporary.replace(target)
        print(f"保存しました: {target} ({target.stat().st_size:,} bytes)")
        return
    if "gdrive_id" in spec:
        try:
            print(f"取得中: {name} (Google Drive)")
            fetch_from_gdrive(spec, temporary)
            verify_download(temporary, spec)
        except Exception as error:
            errors.append(f"Google Drive: {error}")
            temporary.unlink(missing_ok=True)
        else:
            temporary.replace(target)
            print(f"保存しました: {target} ({target.stat().st_size:,} bytes)")
            return
    raise RuntimeError(
        f"取得に失敗しました: {name}。\n" + "\n".join(errors) + f"\n{MANUAL_HINT}"
    )


for name, spec in ANNOTATION_SOURCES.items():
    fetch_file(name, spec, ANNOTATIONS_DIR, annotation_is_usable)

missing = [name for name, spec in ARCHIVE_SOURCES.items()
           if not archive_is_usable(ARCHIVE_DIR / name, spec)]
if missing:
    needed = sum(ARCHIVE_SOURCES[name]["size"] for name in missing)
    ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
    free = shutil.disk_usage(ARCHIVE_DIR).free
    print(f"画像ZIP {len(missing)}個（合計 {needed / 1024**3:.1f} GiB）を取得します。")
    print(f"Driveの空き容量: {free / 1024**3:.1f} GiB")
    if free < needed + 512 * 1024**2:
        raise RuntimeError(
            "Driveの空き容量が不足しています。不要なファイルを削除するか、"
            "別のアカウントのDriveを使ってください。"
        )
    for name in missing:
        fetch_file(name, ARCHIVE_SOURCES[name], ARCHIVE_DIR, archive_is_usable)
else:
    print("画像ZIPは取得済み:")
    for name in ARCHIVE_SOURCES:
        path = ARCHIVE_DIR / name
        print(f"  {path} ({path.stat().st_size:,} bytes)")


## 4. 画像の展開と2クラスのYOLOラベル生成

元のtrain・val分割を維持し、次のルールで変換します。

- `tag=person` かつ `extra.ignore` が未指定または0の人物を対象にします。
- `0: person` はvbox、`1: head` はhboxです。頭部はさらに `head_attr.ignore` が未指定または0のものを使います。
- 座標 `[x, y, w, h]` を画像内にクリップし、YOLOの正規化座標に変換します。欠損・非有限値・幅高さが0以下・画像内の面積が0の枠は除きます。
- 空ラベルの画像も残します。除外数と各クラスの枠数は処理後に表示し、JSONにも保存します。

**無視枠をラベルから除いても、学習時にその領域の損失が無効になるわけではありません。** この出力は通常のYOLO用データで、CrowdHuman公式の無視領域対応や公式評価の実装ではありません。元注釈はDriveに保持します。
注釈形式の出典: [CrowdHuman公式仕様](https://www.crowdhuman.org/download.html)

ZIPは1個ずつローカルへコピーし、注釈に対応する画像だけ展開します。初回展開時はZIP読み取りのCRCチェックが働き、変換時は全画像をデコードします。同じ設定で再実行すると、サイズが一致する展開済み画像を再利用し、ラベルを再生成します。ZIP全体のハッシュ照合は行いません。
完了後、`images/{train,val}`、`labels/{train,val}`、`data.yaml`、`preparation_report.json` が出力されます。

**完了時の集計結果を共有してください。学習は実行しません。**


In [ ]:
from pathlib import Path
import json
import hashlib
import math
import shutil
import tempfile
import zipfile
from collections import Counter
from PIL import Image
from tqdm.auto import tqdm


def read_records(path):
    with path.open(encoding="utf-8") as stream:
        for line_number, line in enumerate(stream, 1):
            if not line.strip():
                continue
            record = json.loads(line)
            image_id = record.get("ID")
            if (not isinstance(image_id, str) or not image_id
                    or "/" in image_id or "\\" in image_id or image_id in (".", "..")):
                raise ValueError(f"不正な画像ID: {path}:{line_number}")
            if not isinstance(record.get("gtboxes"), list):
                raise ValueError(f"gtboxesがありません: {path}:{line_number}")
            yield record


def normalized_box(box, width, height):
    if not isinstance(box, (list, tuple)) or len(box) != 4:
        return None
    try:
        x, y, w, h = map(float, box)
    except (TypeError, ValueError):
        return None
    if not all(math.isfinite(v) for v in (x, y, w, h)) or w <= 0 or h <= 0:
        return None
    left, top = max(0.0, x), max(0.0, y)
    right, bottom = min(float(width), x + w), min(float(height), y + h)
    if right <= left or bottom <= top:
        return None
    return ((left + right) / (2 * width), (top + bottom) / (2 * height),
            (right - left) / width, (bottom - top) / height)


def labels_for_record(record, width, height):
    lines, counts = [], Counter()
    for obj in record["gtboxes"]:
        if obj.get("tag") != "person":
            counts["excluded_non_person"] += 1
            continue
        if (obj.get("extra") or {}).get("ignore", 0) != 0:
            counts["excluded_person_ignore"] += 1
            continue
        for class_id, key in ((0, "vbox"), (1, "hbox")):
            if class_id == 1 and (obj.get("head_attr") or {}).get("ignore", 0) != 0:
                counts["excluded_head_ignore"] += 1
                continue
            box = normalized_box(obj.get(key), width, height)
            if box is None:
                counts[f"invalid_{key}"] += 1
                continue
            lines.append(f"{class_id} " + " ".join(f"{value:.10f}" for value in box))
            counts[key] += 1
    return lines, counts


def atomic_text(path, text):
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(text, encoding="utf-8")
    temporary.replace(path)


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def prepare_dataset(data_dir, output_dir):
    annotation_paths = {
        split: data_dir / "raw" / "annotations" / f"annotation_{split}.odgt"
        for split in ("train", "val")
    }
    archive_paths = {
        "train": [data_dir / "raw" / "archives" / f"CrowdHuman_train{i:02d}.zip"
                  for i in (1, 2, 3)],
        "val": [data_dir / "raw" / "archives" / "CrowdHuman_val.zip"],
    }
    source_paths = list(annotation_paths.values()) + sum(archive_paths.values(), [])
    for path in source_paths:
        if not path.is_file() or path.stat().st_size == 0:
            raise FileNotFoundError(f"必要なファイルがないか空です: {path}")

    config = {
        "schema": 1,
        "classes": {"0": "person_vbox", "1": "head_hbox"},
        "policy": "person_tag; extra.ignore=0; head_attr.ignore=0; clip; discard_invalid",
        "annotations": {split: sha256_file(path) for split, path in annotation_paths.items()},
        # ZIPはサイズと更新時刻を記録します。全体のハッシュ検証ではありません。
        "archives": {str(path): {"size": path.stat().st_size, "mtime_ns": path.stat().st_mtime_ns}
                     for path in sum(archive_paths.values(), [])},
    }
    marker = output_dir / ".build_config.json"
    if output_dir.exists() and any(output_dir.iterdir()):
        if not marker.is_file() or json.loads(marker.read_text()) != config:
            raise RuntimeError("出力先に別設定または管理対象外のデータがあります。OUTPUT_DIRを別の空フォルダに変更してください。")
    output_dir.mkdir(parents=True, exist_ok=True)
    atomic_text(marker, json.dumps(config, ensure_ascii=False, indent=2))

    ids = {}
    for split, path in annotation_paths.items():
        ids[split] = set()
        for record in read_records(path):
            if record["ID"] in ids[split]:
                raise ValueError(f"注釈の画像ID重複: {split}/{record['ID']}")
            ids[split].add(record["ID"])
        if not ids[split]:
            raise ValueError(f"注釈が空です: {path}")
        for kind in ("images", "labels"):
            (output_dir / kind / split).mkdir(parents=True, exist_ok=True)
    if ids["train"] & ids["val"]:
        raise ValueError("trainとvalで画像IDが重複しています。")

    # ZIPの目録を照合します。全画像の展開前に欠落・重複を検出します。
    plans, required_bytes, largest_zip = [], 0, 0
    for split, paths in archive_paths.items():
        found = set()
        for path in paths:
            entries = []
            with zipfile.ZipFile(path) as archive:
                for member in archive.infolist():
                    name = Path(member.filename)
                    if member.is_dir() or name.suffix.lower() != ".jpg" or name.stem not in ids[split]:
                        continue
                    if name.stem in found:
                        raise ValueError(f"ZIP内の画像ID重複: {split}/{name.stem}")
                    found.add(name.stem)
                    target = output_dir / "images" / split / f"{name.stem}.jpg"
                    if not target.is_file() or target.stat().st_size != member.file_size:
                        required_bytes += member.file_size
                    entries.append((member.filename, target, member.file_size))
            plans.append((path, entries))
            largest_zip = max(largest_zip, path.stat().st_size)
        missing = ids[split] - found
        if missing:
            raise ValueError(f"{split}: ZIPに画像が{len(missing)}件不足。例: {sorted(missing)[:5]}")
        print(f"{split}: 注釈とZIPの画像ID一致 {len(found):,}件")

    free = shutil.disk_usage(output_dir).free
    needed = required_bytes + largest_zip + 2 * 1024**3
    print(f"ローカル空き: {free / 1024**3:.2f} GiB / 必要目安: {needed / 1024**3:.2f} GiB")
    if free < needed:
        raise RuntimeError("ローカル空き容量が不足しています。Driveの元データは変更していません。")

    # 一度に1個だけZIPをローカルへコピーし、必要な画像だけ取り出します。
    # 一時領域のみ自動削除。出力画像は一時ファイル経由で確定します。
    with tempfile.TemporaryDirectory(prefix="crowdhuman_zip_", dir=output_dir.parent) as cache:
        for source, entries in plans:
            pending = [(name, target, size) for name, target, size in entries
                       if not target.is_file() or target.stat().st_size != size]
            if not pending:
                print(f"{source.name}: 展開済みのためスキップ")
                continue
            local_zip = Path(cache) / source.name
            print(f"ローカルへコピー中: {source.name}")
            shutil.copyfile(source, local_zip)
            with zipfile.ZipFile(local_zip) as archive:
                for name, target, size in tqdm(pending, desc=source.name):
                    temporary = target.with_name(target.name + ".part")
                    with archive.open(name) as src, temporary.open("wb") as dst:
                        shutil.copyfileobj(src, dst, length=1024 * 1024)
                    if temporary.stat().st_size != size:
                        raise IOError(f"画像サイズ不一致: {name}")
                    temporary.replace(target)
            local_zip.unlink()

    summary = {}
    for split, annotation_path in annotation_paths.items():
        counts = Counter()
        for record in tqdm(read_records(annotation_path), total=len(ids[split]), desc=f"{split} ラベル変換"):
            image_path = output_dir / "images" / split / f"{record['ID']}.jpg"
            # 全画像をデコードし、破損や読み取り不能を検出します。
            with Image.open(image_path) as image:
                image.load()
                width, height = image.size
            lines, box_counts = labels_for_record(record, width, height)
            counts.update(box_counts)
            counts["images"] += 1
            counts["empty_labels"] += int(not lines)
            label_path = output_dir / "labels" / split / f"{record['ID']}.txt"
            atomic_text(label_path, "\n".join(lines) + ("\n" if lines else ""))
        summary[split] = dict(counts)
        if counts["vbox"] == 0 or counts["hbox"] == 0:
            raise ValueError(f"{split}: personまたはheadの有効ラベルが0件です。")
        if len(list((output_dir / "images" / split).glob("*.jpg"))) != len(ids[split]):
            raise ValueError(f"{split}: 出力先に余分な画像があります。別の空フォルダを指定してください。")
        if len(list((output_dir / "labels" / split).glob("*.txt"))) != len(ids[split]):
            raise ValueError(f"{split}: 出力ラベル数が一致しません。")

    yaml_path = output_dir / "data.yaml"
    yaml_text = (f"path: {json.dumps(str(output_dir.resolve()))}\n"
                 "train: images/train\nval: images/val\nnames:\n  0: person\n  1: head\n")
    atomic_text(yaml_path, yaml_text)
    atomic_text(output_dir / "preparation_report.json", json.dumps(summary, ensure_ascii=False, indent=2))
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    print(f"準備完了。学習用YAML: {yaml_path}")
    return yaml_path

DATA_YAML = prepare_dataset(DATA_DIR, OUTPUT_DIR)


## 5. Ultralyticsの導入と事前学習モデルの取得

元ノートブックと同じ、COCO事前学習済みの `yolo26n.pt` を使用します。
再実行・再開時に同じUltralyticsを使うため、バージョンを `8.4.0` に固定します。
初回はインターネットから取得して読み込みを確認した後、Driveの `crowdhuman_vbox_ft/weights` に保存します。保存済みの場合は再利用します。
重みのSHA-256とPyTorch・CUDA・Ultralyticsのバージョンを `pretrained_environment.json` に記録します。SHA-256はコピーの整合性確認と記録のためのもので、配布元の署名検証ではありません。

この段階ではモデルの検出ヘッドはCOCO用です。person・headの2クラスへの変更は、セル6の学習処理が `data.yaml` に基づいて行います。学習・推論は実行しません。

依存パッケージの変更に伴って再起動を求められた場合は、ランタイムを再起動して上から実行してください。Colabローカルの画像が残っていれば、セル4で再利用できます。

参照: [YOLO26公式ドキュメント](https://docs.ultralytics.com/models/yolo26/)、[固定バージョンの配布情報](https://pypi.org/project/ultralytics/8.4.0/)

モデルを取得できたらセル6へ進み、学習条件を設定してください。


In [ ]:
import importlib.metadata
import json
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

ULTRALYTICS_VERSION = "8.4.0"
MODEL_NAME = "yolo26n.pt"
WEIGHTS_DIR = Path("/content/drive/MyDrive/crowdhuman_vbox_ft/weights")
PRETRAINED_PATH = WEIGHTS_DIR / MODEL_NAME

if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("先にセル2でGoogle Driveを接続してください。")
if not Path(DATA_YAML).is_file():
    raise RuntimeError("先にセル3〜4でデータセットを準備してください。")

loaded = sys.modules.get("ultralytics")
if loaded is not None and loaded.__version__ != ULTRALYTICS_VERSION:
    raise RuntimeError("別バージョンのUltralyticsが読み込まれています。ランタイムを再起動してから実行してください。")
try:
    installed_version = importlib.metadata.version("ultralytics")
except importlib.metadata.PackageNotFoundError:
    installed_version = None
if installed_version != ULTRALYTICS_VERSION:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", f"ultralytics=={ULTRALYTICS_VERSION}"],
        check=True,
    )

import torch
import ultralytics
from ultralytics import YOLO
from ultralytics.utils.downloads import safe_download

if ultralytics.__version__ != ULTRALYTICS_VERSION:
    raise RuntimeError("バージョンが反映されていません。ランタイムを再起動してください。")

WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
if PRETRAINED_PATH.exists():
    if not PRETRAINED_PATH.is_file() or PRETRAINED_PATH.stat().st_size == 0:
        raise RuntimeError(f"保存済み重みが不正です。ファイルを確認してください: {PRETRAINED_PATH}")
    print(f"Driveの保存済み重みを使用: {PRETRAINED_PATH}")
    model = YOLO(str(PRETRAINED_PATH), task="detect")
else:
    # ローカルで取得・読み込みを確認してからDriveへコピーします。
    url = f"https://github.com/ultralytics/assets/releases/download/v8.4.0/{MODEL_NAME}"
    with tempfile.TemporaryDirectory(prefix="crowdhuman_model_") as temporary:
        local_path = Path(temporary) / MODEL_NAME
        safe_download(url=url, file=str(local_path), unzip=False, min_bytes=100_000)
        if not local_path.is_file() or local_path.stat().st_size < 100_000:
            raise RuntimeError("公式配布先から重みを取得できませんでした。時間を置いて再実行してください。")
        model = YOLO(str(local_path), task="detect")
        partial_path = PRETRAINED_PATH.with_suffix(".pt.part")
        shutil.copyfile(local_path, partial_path)
        if sha256_file(local_path) != sha256_file(partial_path):
            raise IOError("Driveへのコピー後に重みのハッシュが一致しません。再実行してください。")
        partial_path.replace(PRETRAINED_PATH)
    # 一時パスをモデルに残さないよう、Driveの確定ファイルを読み直します。
    model = YOLO(str(PRETRAINED_PATH), task="detect")

if model.task != "detect":
    raise ValueError("物体検出モデルではありません。")
environment = {
    "ultralytics": ultralytics.__version__,
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "model": MODEL_NAME,
    "weights_sha256": sha256_file(PRETRAINED_PATH),
}
atomic_text(WEIGHTS_DIR / "pretrained_environment.json", json.dumps(environment, ensure_ascii=False, indent=2))
print(json.dumps(environment, ensure_ascii=False, indent=2))
print(f"事前学習重み: {PRETRAINED_PATH}")
print(f"学習用データ: {DATA_YAML}")
model.info()
print("モデル取得完了。現在の検出ヘッドはCOCO用です。2クラスへの変更は学習開始時に行います。")


## 6. 学習・エポックごとのDrive自動保存・再開

**実行すると学習が開始します。** 冒頭のEPOCHS・IMGSZ・BATCH・WORKERS・RUN_NAMEを設定してください。
学習条件は従来どおり、AdamW（初期学習率0.001）、AMP有効、早期終了なしです。ウォームアップなども変更していません。
評価時はEMAのBatchNorm統計がFP16変換でInfになることを防ぐためFP32を使い、保存時は型変換せず、元のデータ型と値を維持します。学習中のAMPとAMP scalerは従来どおりです。
EPOCHSは追加回数ではなく総エポック数です。同じRUN_NAMEで再開する場合は同じ設定を維持してください。

各エポック終了時に `/content/drive/MyDrive/crowdhuman_vbox_ft/runs/<RUN_NAME>/epochs/epoch_000001` などへ保存します。

- `checkpoint.pt`: 学習モデル状態・EMA・オプティマイザー・AMP scaler（保存時にデータ型を変換しません）を含む再開用重み
- `best.pt`・`results.csv`・`args.yaml`・`snapshot.json`: best重み、累積指標、設定、SHA-256付き保存情報
- 実験フォルダの `latest.json`: 全ファイルのコピーとハッシュ確認が完了した最後のエポック
- `final/`: 正常終了時の推論用重み・CSV・図など

損失および保存する重み・オプティマイザー状態にNaN/Infがあれば停止します。エラーの `保存開始前` はコピー前の元状態、`保存用コピー` はコピー後の状態を表します。検出したテンソル名・dtype・shape・NaN/Infの個数も表示します。保存失敗時にも `latest.json` を進めません。
再実行すると、最後に正常保存されたエポックの次から再開します。学習モデルとEMAを別々に復元し、オプティマイザー・AMP scalerも引き継ぎます。
保存無効版で作成したローカル出力と混在させないよう、自動保存を使う最初の実験には新しいRUN_NAMEを指定してください。
データや設定の変更も新しいRUN_NAMEで開始してください。再開前にはハッシュ・エポック数・保存済みテンソルの値を検査します。
旧形式のチェックポイントも検査後に読み込みますが、FP16保存で失われた精度は復元できません。旧形式からNaNが再発する場合は正常な保存分または新しいRUN_NAMEが必要です。

ランタイム消失後はセル1〜5で環境・データを復元してからこのセルを実行してください。
完了済みの場合は再学習しません。final保存前に中断した場合も最終エポックのチェックポイントは使用できます。
ローカルログは `TRAINING.save_dir`、Driveの保存済みログはセル7でダウンロードできます。
各エポックを保持するため、Driveの使用容量は増加します。エポック途中からの再開や中断なしの学習との完全な数値一致は保証しません。

旧版で既にEMAがInfになったメモリ上のTRAININGは再利用せず、更新したセル6でTrainerを作り直してください。正常な保存分があればそこから再開し、なければ1エポック目から開始します。異常なテンソルを0や上限値で置換する処理は行いません。


In [ ]:
# 初回学習前に変更してください。同じRUN_NAMEでの再開時は設定を維持します。
EPOCHS = 100
IMGSZ = 640
BATCH = 8
WORKERS = 2
RUN_NAME = "yolo26n_person_head"

import hashlib
import json
import math
from copy import deepcopy
from datetime import datetime
import shutil
from pathlib import Path

import torch
import ultralytics
import yaml
from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.utils.torch_utils import unwrap_model

DRIVE_RUNS_DIR = Path("/content/drive/MyDrive/crowdhuman_vbox_ft/runs")
LOCAL_RUNS_DIR = Path("/content/crowdhuman_training")


def copy_verified(source, target):
    target.parent.mkdir(parents=True, exist_ok=True)
    partial = target.with_name(target.name + ".part")
    shutil.copyfile(source, partial)
    digest = sha256_file(source)
    if sha256_file(partial) != digest:
        raise IOError(f"コピーのハッシュ不一致: {target}")
    partial.replace(target)
    return digest


def check_finite_state(value, name):
    if isinstance(value, torch.nn.Module):
        value = value.state_dict()
    if isinstance(value, torch.Tensor):
        finite = torch.isfinite(value).all().item()
    elif isinstance(value, float):
        finite = math.isfinite(value)
    else:
        if isinstance(value, dict):
            for key, item in value.items():
                check_finite_state(item, f"{name}.{key}")
        elif isinstance(value, (list, tuple)):
            for index, item in enumerate(value):
                check_finite_state(item, f"{name}[{index}]")
        return
    if not finite:
        detail = ""
        if isinstance(value, torch.Tensor):
            detail = (f" dtype={value.dtype}, shape={tuple(value.shape)},"
                      f" NaN={torch.isnan(value).sum().item()},"
                      f" Inf={torch.isinf(value).sum().item()}")
        raise FloatingPointError(
            f"{name} にNaN/Infを検出しました。{detail} 保存・再開を停止します。"
        )


def check_training_loss(trainer):
    for name in ("loss", "loss_items", "tloss"):
        value = getattr(trainer, name, None)
        if value is not None:
            check_finite_state(value, f"epoch {trainer.epoch + 1}.学習損失.{name}")


def finalize_checkpoint(path, train_results=None):
    # 標準のstrip_optimizerはFP16化するため使わず、元のdtypeを維持します。
    path = Path(path)
    checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    model = checkpoint.get("ema")
    if model is None:
        model = checkpoint["model"]
    model = model.eval()
    if hasattr(model, "criterion"):
        model.criterion = None
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    check_finite_state(model, "最終保存前.model")
    checkpoint["model"] = model
    for key in ("ema", "optimizer", "scaler", "updates", "best_fitness"):
        checkpoint[key] = None
    checkpoint.pop("training_model_state", None)
    checkpoint.pop("checkpoint_schema", None)
    checkpoint["epoch"] = -1
    if train_results is not None:
        checkpoint["train_results"] = train_results
    partial = path.with_name(path.name + ".inference.part")
    torch.save(checkpoint, partial)
    partial.replace(path)
    return checkpoint.get("train_results")


class DtypePreservingCheckpointTrainer(DetectionTrainer):
    def validate(self):
        # 8.4.0のValidatorはtrainer.ampから評価精度を決め、EMAを直接half()化します。
        # 学習用AMP/scaler/args.ampは維持し、評価呼び出し中だけFP32を指定します。
        training_amp = self.amp
        try:
            self.amp = False
            return super().validate()
        finally:
            self.amp = training_amp

    def final_eval(self):
        train_results = None
        if self.last.exists():
            train_results = finalize_checkpoint(self.last)
        if self.best.exists():
            finalize_checkpoint(self.best, train_results=train_results)
            self.validator.args.half = False
            self.validator.args.plots = self.args.plots
            self.validator.args.compile = False
            self.metrics = self.validator(model=self.best)
            self.metrics.pop("fitness", None)
            self.run_callbacks("on_fit_epoch_end")

    def save_model(self):
        # 標準のFP16変換を使わず、学習中の状態をそのまま保存します。
        check_training_loss(self)
        # コピー前に検査し、元の状態と保存用コピーの異常を区別します。
        # state_dict()の値は参照するだけで、補正やdtype変更は行いません。
        live_state = {
            "training_model_state": unwrap_model(self.model).state_dict(),
            "ema": unwrap_model(self.ema.ema).state_dict(),
            "optimizer": self.optimizer.state_dict(),
            "scaler": self.scaler.state_dict(),
        }
        for key, value in live_state.items():
            check_finite_state(value, f"保存開始前.{key}")
        checkpoint = {
            "epoch": self.epoch, "best_fitness": self.best_fitness,
            "model": None,
            "ema": deepcopy(unwrap_model(self.ema.ema)),
            "training_model_state": deepcopy(unwrap_model(self.model).state_dict()),
            "updates": self.ema.updates,
            "optimizer": deepcopy(self.optimizer.state_dict()),
            "scaler": deepcopy(self.scaler.state_dict()),
            "train_args": vars(self.args).copy(),
            "train_metrics": {**self.metrics, "fitness": self.fitness},
            "train_results": self.read_results_csv(),
            "date": datetime.now().isoformat(), "version": ultralytics.__version__,
            "resume_precision": "preserved", "checkpoint_schema": 2,
        }
        for key in ("training_model_state", "ema", "optimizer", "scaler"):
            check_finite_state(checkpoint[key], f"保存用コピー.{key}")
        self.last.parent.mkdir(parents=True, exist_ok=True)
        partial = self.last.with_name(self.last.name + ".part")
        torch.save(checkpoint, partial)
        partial.replace(self.last)
        if self.best_fitness == self.fitness:
            copy_verified(self.last, self.best)


def create_training(data_yaml, pretrained_path, drive_runs, local_runs,
                 run_name, epochs, imgsz, batch, workers, *, device=0, amp=True,
                 plots=True, require_cuda=True):
    if ultralytics.__version__ != "8.4.0":
        raise RuntimeError("セル5でUltralytics 8.4.0を導入してください。")
    if require_cuda and not torch.cuda.is_available():
        raise RuntimeError("GPUが利用できません。Colabのランタイム設定を確認してください。")
    if type(epochs) is not int or epochs < 1:
        raise ValueError("EPOCHSは1以上の整数にしてください。")
    if type(batch) is not int or batch < 1:
        raise ValueError("BATCHは1以上の整数にしてください。")
    if not run_name or Path(run_name).name != run_name or run_name in (".", "..") or "\\" in run_name:
        raise ValueError("RUN_NAMEには単一のフォルダ名を指定してください。")
    data_yaml, pretrained_path = Path(data_yaml), Path(pretrained_path)
    dataset_dir = data_yaml.parent
    dataset_config = dataset_dir / ".build_config.json"
    dataset_report = dataset_dir / "preparation_report.json"
    for path in (data_yaml, pretrained_path, dataset_config, dataset_report):
        if not path.is_file():
            raise FileNotFoundError(f"先にセル3〜5を実行してください: {path}")
    names = yaml.safe_load(data_yaml.read_text())["names"]
    if names != {0: "person", 1: "head"}:
        raise ValueError("data.yamlのクラスがperson・headの2クラスではありません。")

    drive_run = Path(drive_runs) / run_name
    local_run = Path(local_runs).resolve() / run_name
    train_args = {
        "data": str(data_yaml.resolve()), "epochs": epochs, "imgsz": imgsz,
        "batch": batch, "workers": workers, "device": device,
        "project": str(local_run.parent), "name": run_name, "exist_ok": True,
        "optimizer": "AdamW", "lr0": 0.001, "seed": 0,
        "patience": 0, "save": True, "save_period": -1,
        "val": True, "plots": plots, "amp": amp, "cache": False,
        "close_mosaic": min(10, epochs),
    }
    config = {
        "schema": 1, "ultralytics": ultralytics.__version__,
        "pretrained_sha256": sha256_file(pretrained_path),
        "dataset_config_sha256": sha256_file(dataset_config),
        "data_yaml_sha256": sha256_file(data_yaml), "train_args": train_args,
    }
    config_text = json.dumps(config, sort_keys=True, ensure_ascii=False)
    config_hash = hashlib.sha256(config_text.encode()).hexdigest()
    for root, marker_name in ((drive_run, "run_config.json"), (local_run, ".run_config.json")):
        marker = root / marker_name
        if root.exists() and any(root.iterdir()):
            if not marker.is_file() or json.loads(marker.read_text()) != config:
                raise RuntimeError(f"別設定または管理対象外の保存先です: {root}。新しいRUN_NAMEを指定してください。")
        root.mkdir(parents=True, exist_ok=True)
        atomic_text(marker, json.dumps(config, ensure_ascii=False, indent=2))

    latest_path = drive_run / "latest.json"
    resume_path = None
    training_model_state = None
    if latest_path.is_file():
        latest = json.loads(latest_path.read_text())
        completed = latest["completed_epochs"]
        if latest.get("config_sha256") != config_hash or type(completed) is not int or not 1 <= completed <= epochs:
            raise ValueError("再開用の管理情報が学習設定と一致しません。")
        snapshot = drive_run / "epochs" / f"epoch_{completed:06d}"
        metadata = json.loads((snapshot / "snapshot.json").read_text())
        if metadata.get("config_sha256") != config_hash or metadata.get("completed_epochs") != completed:
            raise ValueError("保存済みエポックの管理情報が一致しません。")
        expected_files = {"checkpoint.pt", "best.pt", "results.csv", "args.yaml"}
        if set(metadata["files"]) != expected_files:
            raise ValueError("保存済みエポックの構成が不正です。")
        for name, digest in metadata["files"].items():
            path = snapshot / name
            if not path.is_file() or sha256_file(path) != digest:
                raise IOError(f"保存済みファイルの検証に失敗しました: {path}")
        checkpoint = torch.load(snapshot / "checkpoint.pt", map_location="cpu", weights_only=False)
        if checkpoint.get("epoch") != completed - 1 or checkpoint.get("optimizer") is None:
            raise ValueError("チェックポイントに再開用の状態がありません。")
        for key in ("model", "ema", "training_model_state", "optimizer", "scaler"):
            check_finite_state(checkpoint.get(key), f"再開前.{key}")
        training_model_state = checkpoint.get("training_model_state")
        if checkpoint.get("checkpoint_schema") == 2 and training_model_state is None:
            raise ValueError("再開用の学習モデル状態がありません。")
        if training_model_state is None:
            print("[WARN] 旧形式の保存状態です。保存時に失われた精度は復元できません。"
                  "再開直後にNaNが再発する場合は、この保存分からの正常な再開を保証できません。")
        del checkpoint
        if completed == epochs:
            print(f"指定した{epochs}エポックは学習・保存済みです: {snapshot}")
            print("再学習は行いません。最終成果物が未作成の場合も、このチェックポイントを利用できます。")
            return None
        resume_path = local_run / "weights" / "last.pt"
        copy_verified(snapshot / "checkpoint.pt", resume_path)
        copy_verified(snapshot / "best.pt", local_run / "weights" / "best.pt")
        copy_verified(snapshot / "results.csv", local_run / "results.csv")
        copy_verified(snapshot / "args.yaml", local_run / "args.yaml")
        print(f"{completed}エポック保存済み。{completed + 1}エポック目から再開します。")
    else:
        # 初回エポックの保存前に中断した場合、未確定のローカルCSVを作り直します。
        uncommitted_csv = local_run / "results.csv"
        if uncommitted_csv.exists():
            uncommitted_csv.replace(local_run / "results.csv.uncommitted")
        print(f"新規学習: {epochs}エポック / 保存先: {drive_run}")

    def save_epoch(trainer):
        check_training_loss(trainer)
        completed = trainer.epoch + 1
        snapshot = drive_run / "epochs" / f"epoch_{completed:06d}"
        files = {
            "checkpoint.pt": Path(trainer.last), "best.pt": Path(trainer.best),
            "results.csv": Path(trainer.csv), "args.yaml": Path(trainer.save_dir) / "args.yaml",
        }
        digests = {name: copy_verified(path, snapshot / name) for name, path in files.items()}
        metadata = {
            "completed_epochs": completed, "config_sha256": config_hash,
            "files": digests, "torch": str(torch.__version__), "cuda": torch.version.cuda,
        }
        atomic_text(snapshot / "snapshot.json", json.dumps(metadata, ensure_ascii=False, indent=2))
        # 全ファイルのコピーとハッシュ確認が終わるまで再開先を更新しません。
        atomic_text(latest_path, json.dumps({
            "completed_epochs": completed, "config_sha256": config_hash,
        }, indent=2))
        print(f"\nDrive保存完了: epoch {completed}/{epochs} → {snapshot}")

    def save_final(trainer):
        # Ultralyticsの最終処理で軽量化された重み・図を、再開用とは別に保存します。
        final_dir = drive_run / "final"
        for path in Path(trainer.save_dir).iterdir():
            if path.is_file() and path.suffix in (".csv", ".yaml", ".png", ".jpg", ".json"):
                copy_verified(path, final_dir / path.name)
        for path in (Path(trainer.last), Path(trainer.best)):
            copy_verified(path, final_dir / "weights" / path.name)
        atomic_text(final_dir / "completed.json", json.dumps({
            "completed_epochs": trainer.epoch + 1, "config_sha256": config_hash,
        }, indent=2))
        print(f"最終成果物の保存完了: {final_dir}")

    def restore_training_model(trainer):
        # 標準再開で復元されるEMAとは別に、optimizerと対になる学習モデルを戻します。
        if training_model_state is not None:
            unwrap_model(trainer.model).load_state_dict(training_model_state, strict=True)

    trainer = DtypePreservingCheckpointTrainer(overrides={
        **train_args, "model": str(resume_path or pretrained_path),
        "resume": str(resume_path) if resume_path else False,
    })
    trainer.add_callback("on_train_batch_end", check_training_loss)
    trainer.add_callback("on_train_epoch_end", check_training_loss)
    trainer.add_callback("on_pretrain_routine_end", restore_training_model)
    trainer.add_callback("on_model_save", save_epoch)
    trainer.add_callback("on_train_end", save_final)
    return trainer


TRAINING = create_training(
    DATA_YAML, PRETRAINED_PATH, DRIVE_RUNS_DIR, LOCAL_RUNS_DIR,
    RUN_NAME, EPOCHS, IMGSZ, BATCH, WORKERS,
)
if TRAINING is not None:
    TRAINING.train()


## 7. 学習ログのアーカイブ・ダウンロード

[参考ノートブックの「学習結果アーカイブ」セル](https://github.com/KasumiMercury/yolo_person_finetuning_colab/blob/main/yolo_person_finetuning.ipynb)に合わせ、次の内容を日本時間の日時付きZIPにまとめます。

- `results_<日時>.csv`: 学習履歴
- `args_<日時>.yaml` / `.txt`: 学習設定と実験名・保存日時・ベースモデル
- `summary_<日時>.json`: 実行エポック数、最終指標、mAP50-95(B)最大エポックとその指標
- ローカルに出力された図

セル6の `TRAINING.save_dir` を参照し、ZIPをブラウザへダウンロードします。DriveへのZIP保存も無効にしています。
このセルのZIPにはログだけを含めます。重みはDriveの実験フォルダに保存されます。サマリーのbestはCSVのmAP50-95(B)最大のエポックです。
NaNで停止した場合も、それまでのログが残っていれば取得できます。
`DOWNLOAD_RUN_DIR` に任意のフォルダを指定すれば、以前Driveに保存した実験のログも取得できます。
ランタイムのリセット後はDriveの実験フォルダを参照し、finalまたは最後に正常保存されたエポックのログを取得します。CSVや設定が欠けている場合は警告を表示し、取得できる内容を保存します。
ブラウザでダウンロードを許可してください。


In [ ]:
# 学習結果アーカイブ
from pathlib import Path
from datetime import datetime, timezone, timedelta
import csv
import json
import math
import shutil
import tempfile
import yaml
from google.colab import files

DOWNLOAD_RUN_NAME = globals().get("RUN_NAME", "yolo26n_person_head")
DOWNLOAD_RUN_DIR = Path(getattr(
    globals().get("TRAINING"), "save_dir",
    Path("/content/drive/MyDrive/crowdhuman_vbox_ft/runs") / DOWNLOAD_RUN_NAME,
))


def archive_training_results(run_dir):
    run_dir = Path(run_dir)
    if not run_dir.is_dir():
        raise FileNotFoundError(f"ログのフォルダが見つかりません。DOWNLOAD_RUN_DIRを確認してください: {run_dir}")
    final_dir = run_dir / "final"
    if (run_dir / "args.yaml").is_file():
        selected_dir = run_dir
    elif all((final_dir / name).is_file() for name in (
        "completed.json", "weights/best.pt", "weights/last.pt",
    )):
        selected_dir = final_dir
    else:
        latest_path = run_dir / "latest.json"
        if not latest_path.is_file():
            raise FileNotFoundError("学習ログがありません。セル6の出力先または以前Driveに保存した実験フォルダを指定してください。")
        completed = json.loads(latest_path.read_text(encoding="utf-8"))["completed_epochs"]
        if type(completed) is not int or completed < 1:
            raise ValueError("latest.jsonの保存済みエポック数が不正です。")
        selected_dir = run_dir / "epochs" / f"epoch_{completed:06d}"
        if not (selected_dir / "snapshot.json").is_file():
            raise FileNotFoundError(f"保存済みエポックが見つかりません: {selected_dir}")
        print(f"最終成果物が未完成のため、保存済みの{completed}エポック目を使用します。")

    archived_at = datetime.now(timezone(timedelta(hours=9)))
    timestamp = archived_at.strftime("%Y%m%d_%H%M%S")
    # 同じ秒に再実行しても既存のアーカイブを上書きしません。
    workspace = Path(tempfile.mkdtemp(prefix=f"archive_{run_dir.name}_{timestamp}_"))
    staging = workspace / "contents"
    staging.mkdir()
    config_path = run_dir / "run_config.json"
    config = json.loads(config_path.read_text(encoding="utf-8")) if config_path.is_file() else {}
    train_args = config.get("train_args", {})
    summary = {
        "timestamp": timestamp,
        "archived_at": archived_at.isoformat(),
        "run_name": run_dir.name,
        "source_dir": str(selected_dir),
        "base_model": globals().get("MODEL_NAME", "yolo26n.pt"),
        "pretrained_sha256": config.get("pretrained_sha256"),
        "data_yaml": train_args.get("data"),
        "classes": {"0": "person", "1": "head"},
        "warnings": [],
    }

    # CSVと学習設定を日時付きの名前で保存します。
    for name, target in (("results.csv", f"results_{timestamp}.csv"),
                         ("args.yaml", f"args_{timestamp}.yaml")):
        source = selected_dir / name
        if source.is_file():
            shutil.copy2(source, staging / target)
            print(f"[OK] {name} -> {target}")
        else:
            message = f"ファイルがありません: {source}"
            summary["warnings"].append(message)
            print(f"[WARN] {message}")

    args_snapshot = staging / f"args_{timestamp}.yaml"
    if args_snapshot.is_file():
        args_dict = yaml.safe_load(args_snapshot.read_text(encoding="utf-8")) or {}
    else:
        args_dict = train_args
    summary["data_yaml"] = args_dict.get("data", summary["data_yaml"])
    with (staging / f"args_{timestamp}.txt").open("w", encoding="utf-8") as stream:
        stream.write(f"# Training args snapshot\n# run_name: {run_dir.name}\n")
        stream.write(f"# archived_at: {archived_at.isoformat()}\n")
        stream.write(f"# base_model: {summary['base_model']}\n\n")
        for key, value in sorted(args_dict.items()):
            stream.write(f"{key}: {value}\n")

    # CSVの列名の空白を取り除き、最終行とmAP50-95(B)最大の行を記録します。
    # ここでのbestはCSVの指標による選択です。
    try:
        with (staging / f"results_{timestamp}.csv").open(encoding="utf-8", newline="") as stream:
            rows = [{key.strip(): value for key, value in row.items() if key is not None}
                    for row in csv.DictReader(stream)]
        if not rows:
            raise ValueError("results.csvに学習履歴がありません。")

        def metrics(row):
            values = {}
            for key, value in row.items():
                if "metrics/" in key or "loss" in key.lower():
                    number = float(value)
                    values[key] = number if math.isfinite(number) else None
            return values

        summary["total_epochs_run"] = len(rows)
        summary["last_epoch"] = int(float(rows[-1]["epoch"]))
        summary["last_metrics"] = metrics(rows[-1])
        map_col = next((key for key in rows[0] if "mAP50-95" in key and "B" in key), None)
        if map_col:
            candidates = [row for row in rows if math.isfinite(float(row[map_col]))]
            if not candidates:
                raise ValueError("mAP50-95(B)に有効な値がありません。")
            best_row = max(candidates, key=lambda row: float(row[map_col]))
            summary["best_metric_column"] = map_col
            summary["best_epoch"] = int(float(best_row["epoch"]))
            summary["best_metrics"] = metrics(best_row)
    except (OSError, ValueError, TypeError, KeyError, csv.Error) as error:
        summary["summary_extraction_error"] = str(error)
        print(f"[WARN] 指標サマリーを完全には作成できませんでした: {error}")

    # 図と保存管理情報も同梱します。全エポックの履歴は含めません。
    for source in selected_dir.rglob("*"):
        if source.is_file() and source.suffix.lower() in (".png", ".jpg", ".jpeg", ".json"):
            target = staging / source.relative_to(selected_dir)
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, target)
    for name in ("run_config.json", "latest.json"):
        if (run_dir / name).is_file():
            shutil.copy2(run_dir / name, staging / name)
    (staging / f"summary_{timestamp}.json").write_text(
        json.dumps(summary, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8",
    )
    archive_zip = Path(shutil.make_archive(
        str(workspace / f"{run_dir.name}_{timestamp}"), "zip", root_dir=staging,
    ))
    print(f"[Archive] {archive_zip} ({archive_zip.stat().st_size / 1024**2:.1f} MiB)")
    print("同梱ファイル:", sorted(str(path.relative_to(staging)) for path in staging.rglob("*") if path.is_file()))
    return archive_zip


RESULTS_ZIP = archive_training_results(DOWNLOAD_RUN_DIR)
files.download(str(RESULTS_ZIP))
